# ⚠️ SIMULATION: Shop the Look A/B test analysis on synthetic data

**Nothing in this notebook is real user data or a real experiment result.** Every user, conversion,
return and latency below is *generated* from the assumptions in the next cell. The notebook exists to
show how the experiment in `docs/AB_TEST.md` would be analysed, and to check the power calculation
with a Monte Carlo run. **Never quote these numbers as results.**

Seed: 42. Re-running reproduces every number exactly.

In [1]:
import json, hashlib
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

import sys; sys.path.insert(0, '..')
from src.viz import style, SERIES, NEUTRAL, TEXT_2
style()
OUT = Path('../results/ab_simulation'); OUT.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(42)

# ---------------- ASSUMPTIONS (synthetic world) ----------------
A = dict(
    days=14, new_users_per_day=100_000,     # matches docs/AB_TEST.md planning inputs
    baseline_atc=0.08,                      # mean user-level ATC conversion in control
    propensity_concentration=10.0,          # users differ: p_user ~ Beta(mean*c, (1-mean)*c)
    search_share=0.60,                      # users who use the search bar ('triggered': could see the entry point)
    adoption_among_triggered=0.20,          # => 12% of all treatment users adopt
    adopter_relative_lift=0.25,             # TRUE effect we bake in: adopters' ATC prob x1.25
    order_given_atc=0.45,                   # ATC users who place an order
    return_rate_control=0.22,               # fashion returns (assumption)
    return_rate_lift_adopters_pp=0.005,     # look-attributed orders return slightly more (+0.5 pp)
    latency_median_ms=650, latency_sigma=0.38,  # server latency for look sessions (lognormal)
    bounce_text=0.30, bounce_look=0.31,
)
A['adoption_overall'] = A['search_share'] * A['adoption_among_triggered']
print(json.dumps(A, indent=2))
print(f"Expected whole-arm relative lift ~ adoption x lift = {A['adoption_overall'] * A['adopter_relative_lift']:.3%}")

{
  "days": 14,
  "new_users_per_day": 100000,
  "baseline_atc": 0.08,
  "propensity_concentration": 10.0,
  "search_share": 0.6,
  "adoption_among_triggered": 0.2,
  "adopter_relative_lift": 0.25,
  "order_given_atc": 0.45,
  "return_rate_control": 0.22,
  "return_rate_lift_adopters_pp": 0.005,
  "latency_median_ms": 650,
  "latency_sigma": 0.38,
  "bounce_text": 0.3,
  "bounce_look": 0.31,
  "adoption_overall": 0.12
}
Expected whole-arm relative lift ~ adoption x lift = 3.000%


## 1. Generate synthetic users and assign arms by hashing
Assignment uses the same deterministic hash rule as the plan (`user_id + salt` → bucket 0–999).

In [2]:
N = A['days'] * A['new_users_per_day']
users = pd.DataFrame({'user_id': np.arange(N), 'day': np.repeat(np.arange(A['days']), A['new_users_per_day'])})
def bucket(uid, salt='stl_v1'):
    return int(hashlib.md5(f'{uid}:{salt}'.encode()).hexdigest()[:8], 16) % 1000
users['arm'] = np.where(np.fromiter((bucket(u) for u in users.user_id), int, N) < 500, 'A', 'B')

m, c = A['baseline_atc'], A['propensity_concentration']
users['p_user'] = rng.beta(m * c, (1 - m) * c, N)                 # heterogeneous propensity
users['pre_atc'] = rng.poisson(users.p_user * 3)                  # 28-day pre-period ATC count (for CUPED)
users['triggered'] = rng.random(N) < A['search_share']
users['adopter'] = (users.arm == 'B') & users.triggered & (rng.random(N) < A['adoption_among_triggered'])
p = users.p_user * np.where(users.adopter, 1 + A['adopter_relative_lift'], 1.0)
users['atc'] = rng.random(N) < np.clip(p, 0, 1)
users['ordered'] = users.atc & (rng.random(N) < A['order_given_atc'])
rr = A['return_rate_control'] + np.where(users.adopter, A['return_rate_lift_adopters_pp'], 0)
users['returned'] = users.ordered & (rng.random(N) < rr)
users.groupby('arm')[['atc', 'triggered', 'adopter', 'ordered']].mean().style.format('{:.4f}')

,atc,triggered,adopter,ordered
arm,,,,
A,0.0800,0.6006,0.0000,0.0356
B,0.0823,0.6002,0.1199,0.0370


## 2. Validity: sample-ratio mismatch (SRM) check

In [3]:
counts = users.arm.value_counts().reindex(['A', 'B'])
chi2, p_srm = stats.chisquare(counts, [N / 2, N / 2])
print(counts.to_dict(), f'chi2={chi2:.3f}, p={p_srm:.4f}', '-> OK' if p_srm >= 0.001 else '-> SRM! stop and debug')

{'A': 699493, 'B': 700507} chi2=0.734, p=0.3915 -> OK


## 3. Primary metric: user-level ATC conversion (SIMULATION)

In [4]:
def two_prop(df, metric='atc'):
    a, b = df[df.arm == 'A'][metric], df[df.arm == 'B'][metric]
    pa, pb, na, nb = a.mean(), b.mean(), len(a), len(b)
    z, p = proportions_ztest([b.sum(), a.sum()], [nb, na])
    se = np.sqrt(pa * (1 - pa) / na + pb * (1 - pb) / nb)
    diff = pb - pa
    # relative lift CI via the delta method on pb/pa
    rel = pb / pa - 1
    se_rel = np.sqrt((pb * (1 - pb) / nb) / pa**2 + (pb**2) * (pa * (1 - pa) / na) / pa**4)
    return dict(n_A=na, n_B=nb, rate_A=pa, rate_B=pb, diff_pp=diff * 100, ci95_pp=[(diff - 1.96 * se) * 100, (diff + 1.96 * se) * 100],
                rel_lift=rel, rel_ci95=[rel - 1.96 * se_rel, rel + 1.96 * se_rel], z=z, p_value=p)

primary = two_prop(users)
triggered = two_prop(users[users.triggered])
pd.DataFrame({'all users (primary)': primary, 'triggered users': triggered}).T

,n_A,n_B,rate_A,rate_B,diff_pp,ci95_pp,rel_lift,rel_ci95,z,p_value
all users (primary),699493,700507,0.079965,0.082336,0.237102,"[0.146636, 0.327568]",0.029651,"[0.018171, 0.041131]",5.136874,0.0
triggered users,420125,420417,0.080152,0.084183,0.403075,"[0.285659, 0.52049]",0.050289,"[0.035275, 0.065302]",6.728241,0.0


### CUPED (pre-registered secondary): variance reduction with the pre-period ATC count

In [5]:
theta = np.cov(users.atc.astype(float), users.pre_atc)[0, 1] / users.pre_atc.var()
users['atc_cuped'] = users.atc - theta * (users.pre_atc - users.pre_atc.mean())
ga, gb = users[users.arm == 'A'].atc_cuped, users[users.arm == 'B'].atc_cuped
diff_c = gb.mean() - ga.mean(); se_c = np.sqrt(ga.var() / len(ga) + gb.var() / len(gb))
se_raw = (primary['ci95_pp'][1] - primary['ci95_pp'][0]) / 2 / 1.96 / 100
cuped = dict(theta=theta, diff_pp=diff_c * 100, ci95_pp=[(diff_c - 1.96 * se_c) * 100, (diff_c + 1.96 * se_c) * 100],
             p_value=2 * stats.norm.sf(abs(diff_c / se_c)), variance_reduction=1 - (se_c / se_raw) ** 2)
cuped

{'theta': np.float64(0.06819925270849175),
 'diff_pp': np.float64(0.23868317947283307),
 'ci95_pp': [np.float64(0.1490690043525999), np.float64(0.32829735459306625)],
 'p_value': np.float64(1.7856730311335853e-07),
 'variance_reduction': np.float64(0.018736719192857354)}

## 4. Guardrails (SIMULATION)

In [6]:
orders = users[users.ordered]
ret_a, ret_b = orders[orders.arm == 'A'].returned, orders[orders.arm == 'B'].returned
d_ret = ret_b.mean() - ret_a.mean(); se_ret = np.sqrt(ret_a.var() / len(ret_a) + ret_b.var() / len(ret_b))
n_look = int(users.adopter.sum())
lat = rng.lognormal(np.log(A['latency_median_ms']), A['latency_sigma'], n_look)
bounce_text = rng.random(int((users.arm == 'A').sum() * A['search_share'])) < A['bounce_text']
bounce_look = rng.random(n_look) < A['bounce_look']
guard = {
  'return_rate_diff_pp': d_ret * 100, 'return_rate_upper95_pp': (d_ret + 1.645 * se_ret) * 100, 'return_threshold_pp': 1.0,
  'p95_latency_ms': float(np.percentile(lat, 95)), 'latency_threshold_ms': 1500,
  'bounce_diff_pp': (bounce_look.mean() - bounce_text.mean()) * 100, 'bounce_threshold_pp': 2.0,
}
guard['return_ok'] = guard['return_rate_upper95_pp'] <= guard['return_threshold_pp']
guard['latency_ok'] = guard['p95_latency_ms'] <= guard['latency_threshold_ms']
guard['bounce_ok'] = guard['bounce_diff_pp'] <= guard['bounce_threshold_pp']
guard

{'return_rate_diff_pp': np.float64(0.04275145258309798),
 'return_rate_upper95_pp': np.float64(0.6483917544396983),
 'return_threshold_pp': 1.0,
 'p95_latency_ms': 1213.1013461457833,
 'latency_threshold_ms': 1500,
 'bounce_diff_pp': np.float64(0.7933949564987275),
 'bounce_threshold_pp': 2.0,
 'return_ok': np.True_,
 'latency_ok': True,
 'bounce_ok': np.True_}

## 5. Novelty check: daily lift (SIMULATION)
The synthetic world has a constant effect, so this shows what a *stable* lift looks like: noisy days scattered around a flat line.

In [7]:
daily = users.groupby(['day', 'arm']).atc.mean().unstack()
daily['rel_lift'] = daily.B / daily.A - 1
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.axhline(0, color=NEUTRAL, linewidth=1)
ax.axhline(primary['rel_lift'], color=TEXT_2, linestyle='--', linewidth=1)
ax.plot(daily.index + 1, daily.rel_lift * 100, marker='o', color=SERIES[0], linewidth=2, markersize=6)
ax.text(14.2, primary['rel_lift'] * 100, f" pooled {primary['rel_lift']:+.1%}", va='center', color=TEXT_2, fontsize=9)
ax.set_xlabel('day of test'); ax.set_ylabel('relative ATC lift, B vs A (%)')
ax.set_title('SIMULATION: daily relative lift in ATC conversion (synthetic data)')
fig.tight_layout(); fig.savefig(OUT / 'simulated_daily_lift.png'); plt.show()

/var/folders/4j/_pc5fncx5_v834mm4ds4wsf40000gn/T/ipykernel_54125/3748813395.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(OUT / 'simulated_daily_lift.png'); plt.show()


## 6. Does the power calculation hold? Monte Carlo check
We repeat a synthetic experiment 2,000 times at the planned sample (203,322 users/arm), with a true +3% relative lift, and count how often p < 0.05. Theory says ≈ 80%. We also run the same check with **no** true effect: it should come out significant ≈ 5% of the time (the false-positive rate).

In [8]:
n_arm, p0 = 203_322, 0.08
sims = 2000
def sim_power(lift):
    xa = rng.binomial(n_arm, p0, sims); xb = rng.binomial(n_arm, p0 * (1 + lift), sims)
    pa, pb = xa / n_arm, xb / n_arm; pp = (xa + xb) / (2 * n_arm)
    z = (pb - pa) / np.sqrt(pp * (1 - pp) * 2 / n_arm)
    return float(np.mean(2 * stats.norm.sf(np.abs(z)) < 0.05))
mc = {'power_at_3pct_lift': sim_power(0.03), 'false_positive_rate_at_0_lift': sim_power(0.0), 'n_per_arm': n_arm, 'simulations': sims}
mc

{'power_at_3pct_lift': 0.78,
 'false_positive_rate_at_0_lift': 0.05,
 'n_per_arm': 203322,
 'simulations': 2000}

## 7. Decision (applying the pre-registered rules in `docs/AB_TEST.md`) on SIMULATED data

In [9]:
sig_pos = primary['ci95_pp'][0] > 0
guards_ok = guard['return_ok'] and guard['latency_ok'] and guard['bounce_ok']
sig_neg = primary['ci95_pp'][1] < 0
decision = 'SHIP' if sig_pos and guards_ok else ('KILL' if sig_neg or not guard['return_ok'] else 'ITERATE')
print(f"[SIMULATION] primary lift {primary['rel_lift']:+.2%} (95% CI {primary['rel_ci95'][0]:+.2%} to {primary['rel_ci95'][1]:+.2%}), p={primary['p_value']:.2g}")
print(f"[SIMULATION] guardrails ok: {guards_ok} -> decision: {decision}")

fig, ax = plt.subplots(figsize=(7, 2.6))
rows = [('All users (primary)', primary), ('Triggered users', triggered)]
for i, (lab, r) in enumerate(rows[::-1]):
    ax.plot(r['rel_ci95'], [i, i], color=SERIES[0], linewidth=3, solid_capstyle='round')
    ax.plot(r['rel_lift'], i, 'o', color=SERIES[0], markersize=8, markeredgecolor='#fcfcfb', markeredgewidth=2)
    ax.text(r['rel_ci95'][1] + 0.002, i, f" {r['rel_lift']:+.1%}", va='center', fontsize=9)
ax.axvline(0, color=NEUTRAL, linewidth=1)
ax.set_yticks(range(len(rows)), [l for l, _ in rows[::-1]]); ax.grid(axis='y', visible=False)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:+.0%}'))
ax.set_xlabel('relative lift in ATC conversion with 95% CI'); ax.set_ylim(-0.6, len(rows) - 0.4)
ax.set_title('SIMULATION: treatment effect estimates (synthetic data)')
fig.tight_layout(); fig.savefig(OUT / 'simulated_lift_ci.png'); plt.show()

def clean(d): return {k: (v.item() if hasattr(v, 'item') else v) for k, v in d.items()}
summary = {'LABEL': 'SIMULATION - synthetic data, not a real experiment', 'assumptions': A,
           'srm': {'counts': counts.to_dict(), 'p': p_srm}, 'primary': clean(primary), 'triggered': clean(triggered),
           'cuped': clean(cuped), 'guardrails': clean(guard), 'monte_carlo_power_check': mc, 'decision': decision}
(OUT / 'SIMULATION_ab_results.json').write_text(json.dumps(summary, indent=2, default=float))
print('saved', OUT / 'SIMULATION_ab_results.json')

[SIMULATION] primary lift +2.97% (95% CI +1.82% to +4.11%), p=2.8e-07
[SIMULATION] guardrails ok: True -> decision: SHIP
saved ../results/ab_simulation/SIMULATION_ab_results.json


/var/folders/4j/_pc5fncx5_v834mm4ds4wsf40000gn/T/ipykernel_54125/715594374.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(OUT / 'simulated_lift_ci.png'); plt.show()


## Takeaways (about the method, not the product)
* With 12% adoption, a +25% effect among adopters is diluted to about +3% across the arm. The **triggered analysis** (users who searched, so they could see the entry point) gives a tighter, larger estimate.
* **CUPED helped very little here (~2% variance reduction)** because the synthetic pre-period ATC count is only weakly correlated with in-test conversion. Its value has to be measured on real pre-period data before we count on it.
* The Monte Carlo check gives ~78% power at the planned sample (target 80%; Monte Carlo SE ≈ 0.9 pp). The gap is consistent with the pooled-variance z-test being slightly more conservative than the unpooled sample-size formula. The false-positive rate is 5.0%, as designed. To guarantee ≥ 80%, round the sample up by ~5%.
* **Again: every number above is synthetic.** Real decisions need the real experiment.